# Open-Source LLMs on Groq — Demo Notebook
For such contents, join linked group https://www.linkedin.com/groups/8944257/ and/or whatsapp https://chat.whatsapp.com/Dc2Tpzzi7wHJgFQdHD2T8B?mode=gi_t

Use **free open-source models** via [Groq](https://groq.com) —  LLM inference engine.


> **Prerequisites:** Get a free `GROQ_API_KEY` at https://console.groq.com/keys and add it to your `.env` file.


## 1. Setup

In [106]:
import os, json, time
from dotenv import load_dotenv
from groq import Groq

load_dotenv()
api_key = os.environ.get("GROQ_API_KEY", "")
if not api_key:
    raise ValueError("GROQ_API_KEY not found in .env — get one free at https://console.groq.com/keys")
if api_key.startswith("xai-"):
    raise ValueError("That's an xAI (Grok) key, not a Groq key. Groq keys start with 'gsk_'.")

client = Groq(api_key=api_key)
print(f"Groq client ready (key: gsk_...{api_key[-4:]})")

# Uncomment and run to install all dependencies:
#!pip install groq python-dotenv ddgs

Groq client ready (key: gsk_...CELr)


## 2. Available Models

All models below are **free** on Groq's developer tier.

| Model ID | Params | Speed | Best For |
|---|---|---|---|
| `llama-3.1-8b-instant` | 8B | ~560 tok/s | Fast drafts, classification, batch jobs |
| `llama-3.3-70b-versatile` | 70B | ~280 tok/s | Complex reasoning, reports — **best all-rounder** |
| `openai/gpt-oss-120b` | 120B | ~500 tok/s | Highest quality open model |
| `openai/gpt-oss-20b` | 20B | ~1000 tok/s | Fastest large model |
| `meta-llama/llama-4-scout-17b-16e-instruct` | 17B MoE | ~750 tok/s | Multimodal (images + text), latest Llama 4 |
| `qwen/qwen3-32b` | 32B | ~400 tok/s | Multilingual (Hindi!), math, reasoning |
| `moonshotai/kimi-k2-instruct` | Large | — | Strong on coding and reasoning |

**Change the variable below to experiment with any model!**

In [107]:
# ╔════════════════════════════════════════════════════════╗
# ║  CHANGE THIS to try different models                  ║
# ╚════════════════════════════════════════════════════════╝
SELECTED_MODEL = "llama-3.3-70b-versatile"

# Fetch live model list from Groq
models = client.models.list()
print(f"Selected: {SELECTED_MODEL}\n")
print(f"{'Model ID':<50} {'Owner':<20} {'Context'}")
print("─" * 80)
for m in sorted(models.data, key=lambda x: x.id):
    ctx = getattr(m, 'context_window', 'N/A')
    marker = " ◄" if m.id == SELECTED_MODEL else ""
    print(f"{m.id:<50} {m.owned_by:<20} {ctx}{marker}")

Selected: llama-3.3-70b-versatile

Model ID                                           Owner                Context
────────────────────────────────────────────────────────────────────────────────
allam-2-7b                                         SDAIA                4096
canopylabs/orpheus-arabic-saudi                    Canopy Labs          4000
canopylabs/orpheus-v1-english                      Canopy Labs          4000
groq/compound                                      Groq                 131072
groq/compound-mini                                 Groq                 131072
llama-3.1-8b-instant                               Meta                 131072
llama-3.3-70b-versatile                            Meta                 131072 ◄
meta-llama/llama-4-scout-17b-16e-instruct          Meta                 131072
meta-llama/llama-prompt-guard-2-22m                Meta                 512
meta-llama/llama-prompt-guard-2-86m                Meta                 512
moonshotai/kimi-k2-instr

---
## 3. Basic Chat Completion

In [108]:
response = client.chat.completions.create(
    model=SELECTED_MODEL,
    messages=[
        {"role": "system", "content": "You are a helpful assistant. Be concise."},
        {"role": "user", "content": "What are the top 3 metrics to evaluate a tech company's financial health?"}
    ],
    temperature=0.5, max_tokens=300,
)
print(response.choices[0].message.content)
print(f"\n[{response.usage.prompt_tokens} in / {response.usage.completion_tokens} out | {response.model}]")

The top 3 metrics to evaluate a tech company's financial health are:

1. **Revenue Growth Rate**: Measures the increase in revenue over time.
2. **Gross Margin**: Indicates the company's profitability by calculating the difference between revenue and cost of goods sold.
3. **Operating Cash Flow**: Shows the company's ability to generate cash from its operations, indicating its financial stability.

[60 in / 79 out | llama-3.3-70b-versatile]


---
## 4. Stock Sentiment Analysis

In [109]:
headlines = [
    "NVIDIA beats Q4 earnings expectations, data center revenue up 93%",
    "NVIDIA announces $50B stock buyback program",
    "Analysts warn NVIDIA's AI growth may slow as competition heats up",
    "China export restrictions could impact NVIDIA's 2026 revenue by 15%",
    "NVIDIA partners with major automakers for autonomous driving chips",
]

prompt = f"""Analyze sentiment of these NVIDIA headlines.
For each: Sentiment (Bullish/Bearish/Neutral), Confidence, brief reason.
End with an overall score from -1.0 (bearish) to +1.0 (bullish).

Headlines:
{chr(10).join(f'{i+1}. {h}' for i, h in enumerate(headlines))}"""

r = client.chat.completions.create(
    model=SELECTED_MODEL,
    messages=[{"role": "system", "content": "Expert financial sentiment analyst."}, {"role": "user", "content": prompt}],
    temperature=0.3, max_tokens=800,
)
print(r.choices[0].message.content)

Here's the sentiment analysis for each headline:

1. **Sentiment: Bullish**, **Confidence: High**, Reason: Beating earnings expectations and significant revenue growth in the data center segment is a strong positive indicator.
2. **Sentiment: Bullish**, **Confidence: High**, Reason: A $50B stock buyback program demonstrates confidence in the company's financials and commitment to returning value to shareholders.
3. **Sentiment: Bearish**, **Confidence: Medium**, Reason: Warning of potential slowing growth in a key segment (AI) due to increasing competition is a negative indicator, but the impact is uncertain.
4. **Sentiment: Bearish**, **Confidence: Medium**, Reason: Potential export restrictions could significantly impact revenue, but the actual impact is uncertain and dependent on various factors.
5. **Sentiment: Bullish**, **Confidence: High**, Reason: Partnering with major automakers for autonomous driving chips is a significant positive development, indicating growing demand and o

---
## 5. Investment Thesis — Reliance Industries

In [110]:
reliance = """
Company: Reliance Industries Ltd (RELIANCE.NS)
Market Cap: ₹20.1 Lakh Crore (~$240B) | P/E: 28 | P/B: 2.8
Revenue: ₹10.1 Lakh Crore (+7.5% YoY) | Net Margin: 8.2% | ROE: 9.1%
Segments: O2C (Petrochemicals), Jio (490M+ subscribers), Retail (18K+ stores), New Energy
Debt/Equity: 0.36 | Promoter Holding: 50.3%
Recent: Jio AI Cloud launch, solar & green hydrogen investments
"""

r = client.chat.completions.create(
    model=SELECTED_MODEL,
    messages=[
        {"role": "system", "content": "Senior Indian equity analyst at a top brokerage."},
        {"role": "user", "content": f"Generate a brief investment thesis for Reliance. Include Rating, Bull Case, Bear Case, Key Catalyst, 1-year target in INR.\n{reliance}"}
    ],
    temperature=0.4, max_tokens=800,
)
print(r.choices[0].message.content)

**Investment Thesis:**
We initiate coverage on Reliance Industries Ltd (RELIANCE.NS) with a **BUY** rating.

**Bull Case:** Reliance's diversified business portfolio, led by the rapidly growing Jio platform, is poised to drive long-term growth. The company's strategic investments in new energy, including solar and green hydrogen, will not only reduce its carbon footprint but also create new avenues for expansion. Additionally, the retail segment is expected to benefit from the increasing demand for omnichannel retail experiences.

**Bear Case:** Intensifying competition in the telecom sector, potential delays in the ramp-up of new energy initiatives, and rising crude oil prices impacting the O2C segment's profitability are key risks to the stock. Furthermore, the company's high debt levels, although manageable, warrant close monitoring.

**Key Catalyst:** The successful rollout of Jio's 5G services, coupled with the expansion of its AI Cloud offerings, is expected to be a significant c

---
## 6. Model Comparison — Same Prompt, 6 Models

See how speed and quality differ across models.

In [111]:
MODELS = [
    "llama-3.1-8b-instant",
    "llama-3.3-70b-versatile",
    "openai/gpt-oss-20b",
    "openai/gpt-oss-120b",
    "qwen/qwen3-32b",
    "meta-llama/llama-4-scout-17b-16e-instruct",
]

prompt = """Tesla: P/E=170, Revenue Growth=+25%, Gross Margin=18.2%, FCF=$3.6B, Beta=2.05.
In exactly 3 bullet points: Is Tesla overvalued?"""

print("MODEL COMPARISON: Is Tesla Overvalued?")
print("=" * 85)

for mid in MODELS:
    try:
        t0 = time.time()
        r = client.chat.completions.create(
            model=mid,
            messages=[{"role": "system", "content": "Concise equity analyst."}, {"role": "user", "content": prompt}],
            temperature=0.3, max_tokens=300,
        )
        dt = time.time() - t0
        tok = r.usage.completion_tokens
        print(f"\n─── {mid}  |  {dt:.2f}s  |  {tok} tok  |  {tok/dt:.0f} tok/s ───")
        print(r.choices[0].message.content)
    except Exception as e:
        print(f"\n[{mid}] Error: {e}")
    time.sleep(1)

MODEL COMPARISON: Is Tesla Overvalued?

─── llama-3.1-8b-instant  |  0.63s  |  222 tok  |  352 tok/s ───
Based on the provided information, here are three points to consider whether Tesla is overvalued:

• **High P/E ratio**: With a P/E ratio of 170, Tesla's stock price is significantly higher than its earnings, which could indicate that the stock is overvalued. This ratio is much higher than the industry average, suggesting that investors are paying a premium for the company's growth prospects.

• **High beta**: Tesla's beta of 2.05 indicates that its stock price is highly volatile and sensitive to market fluctuations. While this can be beneficial in a rising market, it also means that the stock price can drop sharply in a downturn, which could lead to significant losses for investors.

• **High valuation metrics relative to FCF**: With a free cash flow (FCF) of $3.6B, Tesla's valuation metrics such as P/E ratio and revenue growth seem high compared to its ability to generate cash. Th

---
## 7. Streaming + JSON Mode

In [112]:
# 7a. Streaming — see tokens arrive in real-time
stream = client.chat.completions.create(
    model=SELECTED_MODEL,
    messages=[{"role": "user", "content": "Compare growth vs value investing in 2026. Keep to 100 words."}],
    temperature=0.5, max_tokens=300, stream=True,
)
print("Streaming: ", end="")
for chunk in stream:
    if chunk.choices[0].delta.content:
        print(chunk.choices[0].delta.content, end="", flush=True)
print("\n")

Streaming: In 2026, growth investing focuses on stocks with high potential for expansion, often in tech and innovation. Value investing targets undervalued stocks with strong fundamentals. Growth stocks tend to be more volatile, while value stocks offer stable returns. With interest rates rising, value investing may gain traction as investors seek stable income. However, growth stocks may still outperform if innovative companies continue to disrupt markets and drive economic growth. Ultimately, a balanced portfolio combining both strategies can provide optimal returns and risk management. Diversification is key to navigating the 2026 market landscape.



In [113]:
# 7b. JSON Mode — structured extraction
r = client.chat.completions.create(
    model=SELECTED_MODEL,
    messages=[
        {"role": "system", "content": "Extract structured data. Respond in valid JSON only."},
        {"role": "user", "content": """Extract: \"Amazon Q4 revenue $187.8B (+10% YoY). AWS $24.2B (+19%). Operating income $13.2B. Guided Q1 $155-160B.\"
Keys: company, ticker, quarter, revenue_b, revenue_growth_pct, aws_revenue_b, operating_income_b, guidance_low_b, guidance_high_b"""}
    ],
    temperature=0.0, max_tokens=300, response_format={"type": "json_object"},
)
print(json.dumps(json.loads(r.choices[0].message.content), indent=2))

{
  "company": "Amazon",
  "ticker": "AMZN",
  "quarter": "Q4",
  "revenue_b": 187.8,
  "revenue_growth_pct": 10,
  "aws_revenue_b": 24.2,
  "operating_income_b": 13.2,
  "guidance_low_b": 155,
  "guidance_high_b": 160
}


---
## 8. Batch Stock Ratings

In [114]:
stocks = {
    "Reliance (India)": "Conglomerate, P/E=28, Revenue Growth=+7.5%, Margin=8.2%, Jio 490M subs",
    "TCS (India)":      "IT Services, P/E=33, Revenue Growth=+4.2%, Margin=19%, 600K employees",
    "HDFC Bank (India)": "Banking, P/E=20, NII Growth=+10%, NPA=1.2%, Largest private bank",
    "NVDA (US)":        "Semiconductors, P/E=55, Revenue Growth=+94%, Margin=74%, AI leader",
    "TSLA (US)":        "EVs, P/E=170, Revenue Growth=+25%, Margin=18%, Beta=2.05",
    "MSFT (US)":        "Cloud/Software, P/E=36, Revenue Growth=+16%, Margin=70%, Azure+AI",
}

for name, data in stocks.items():
    r = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "system", "content": "In one sentence: Buy/Hold/Sell with the key reason."},
            {"role": "user", "content": f"{name}: {data}"}
        ],
        temperature=0.2, max_tokens=100,
    )
    print(f"{name}: {r.choices[0].message.content}\n")

Reliance (India): Buy: Reliance has a strong presence in various sectors, a large subscriber base in Jio, and a relatively high revenue growth rate, making it an attractive investment opportunity.

TCS (India): Buy: Despite a high P/E ratio, TCS has a strong track record of revenue growth, high margins, and a large employee base, indicating a stable and scalable business model.

HDFC Bank (India): Buy: Strong growth prospects, low NPA ratio, and a leading position in the Indian banking sector.

NVDA (US): Buy: Strong revenue growth, high margin, and leadership in AI technology make NVDA a promising investment opportunity despite its high P/E ratio.

TSLA (US): Sell: The extremely high P/E ratio of 170 indicates overvaluation, despite strong revenue growth and margin.

MSFT (US): Buy: Strong fundamentals, high revenue growth, and increasing presence in the cloud and AI markets make MSFT a solid long-term investment opportunity.



---
## 9. Beyond Finance — Fun Use Cases

In [115]:
# 9a. Cricket Match Analysis 🏏

match = """
Champions Trophy Final: India vs New Zealand, Dubai
India: 265/8 (50 ov) — Rohit 72(80), Kohli 56(65), Pandya 48*(32)
NZ Bowling: Santner 3/42, Boult 2/48, Ferguson 2/58
NZ: 241 all out (48.3 ov) — Williamson 89(102), Ravindra 43(51)
India Bowling: Bumrah 4/38, Kuldeep 2/45, Jadeja 2/41
India won by 24 runs. Bumrah: Player of the Match.
"""

r = client.chat.completions.create(
    model=SELECTED_MODEL,
    messages=[
        {"role": "system", "content": "You are Harsha Bhogle. Analyze with passion and insight. Keep it to 150 words."},
        {"role": "user", "content": f"Post-match analysis — turning point and key performances:\n{match}"}
    ],
    temperature=0.6, max_tokens=400,
)
print("🏏 CRICKET ANALYSIS\n")
print(r.choices[0].message.content)

🏏 CRICKET ANALYSIS

"What a thrilling final we've just witnessed in Dubai. The turning point, for me, was when Jasprit Bumrah dismissed Kane Williamson, breaking the backbone of New Zealand's chase. Bumrah's 4/38 was a masterclass in death bowling. Rohit Sharma's 72 and Virat Kohli's 56 set the tone for India, while Hardik Pandya's cameo provided the late surge. But it was Bumrah's brilliance that sealed the deal. He's the Player of the Match, and rightly so. India's bowling unit was exceptional, but Bumrah's precision and variation made all the difference. A fantastic win for India, and a proud moment for Indian cricket!"


In [116]:
# 9b. Startup Pitch — Shark Tank India Style 🚀

r = client.chat.completions.create(
    model=SELECTED_MODEL,
    messages=[
        {"role": "system", "content": "You are a Shark Tank India judge. Sharp, actionable evaluation. Keep to 200 words."},
        {"role": "user", "content": """Evaluate my startup idea:
App connecting home cooks with people wanting ghar ka khana (like Uber for home food).
Target: Working professionals, PG students, elderly living alone.
Revenue: 15% commission + meal plan subscriptions.
I'm a Bangalore college student with ₹2L savings and basic coding skills.
What's good, what's missing, would you invest?"""}
    ],
    temperature=0.6, max_tokens=500,
)
print("🚀 STARTUP EVALUATION\n")
print(r.choices[0].message.content)

🚀 STARTUP EVALUATION

I like the concept, it's a unique twist on food delivery. The target market is sizable, and the revenue model is straightforward. However, I have concerns about execution. As a solo founder with basic coding skills, scaling the platform and ensuring quality control will be challenging.

What's missing is a clear plan for supply chain management, food safety, and logistics. How will you vet home cooks, ensure consistent quality, and handle delivery? You'll also need to consider regulatory compliance and potential health department issues.

I'd invest ₹10L for 20% equity, but only if you can demonstrate a clear plan to address these concerns and scale the business. I'd also want to see a more robust tech platform and a team in place to support growth. With ₹2L, you can start small, but to make a meaningful impact, you'll need more resources. Are you ready to take on the challenge?


In [117]:
# 9c. Exam Prep — Concepts Made Simple 📚

r = client.chat.completions.create(
    model=SELECTED_MODEL,
    messages=[
        {"role": "system", "content": "Expert teacher who explains using everyday Indian analogies. 3-4 sentences per concept max."},
        {"role": "user", "content": """Explain simply for a B.Tech/MBA student:
1. What is P/E ratio? (use chai stall analogy)
2. TCP vs UDP difference? (use real-life analogy)
3. What is gradient descent? (explain like ordering on Swiggy)"""}
    ],
    temperature=0.5, max_tokens=600,
)
print("📚 CONCEPTS MADE SIMPLE\n")
print(r.choices[0].message.content)

📚 CONCEPTS MADE SIMPLE

Here are the explanations:

1. The P/E ratio is like the price of a chai at a stall. Imagine you're buying a chai stall, and it makes a profit of Rs. 10 per day. If you're paying Rs. 100 for the stall, the P/E ratio is 10 (100/10), meaning you're paying 10 times the daily profit. A higher P/E ratio means you're paying more for each unit of profit.

2. TCP is like sending a registered letter, where you get an acknowledgement that it's delivered, whereas UDP is like sending a postcard, where you just send it and hope it reaches. In TCP, if the packet doesn't reach, it's resent, ensuring reliable delivery, but it's slower. In UDP, packets may get lost, but it's faster, like a quick postcard.

3. Gradient descent is like ordering food on Swiggy - you have an idea of what you want (your target), but you're not sure which restaurant or menu item is best. You start with a guess, get feedback (the 'rating' of your order), and adjust your guess until you find the perfect

---
## 10. Web-Grounded Analysis — DuckDuckGo + LLM

Search the web, then feed results to the LLM for grounded answers.

In [118]:
from ddgs import DDGS

def search_and_analyze(query, num_results=5):
    """Search DuckDuckGo, then ask the LLM to analyze results."""
    print(f'Searching: "{query}"\n')
    results = list(DDGS().text(query, max_results=num_results))
    if not results:
        print("No results found."); return

    context = ""
    for i, r in enumerate(results, 1):
        print(f"{i}. {r.get('title', '')}")
        print(f"   {r.get('body', '')[:100]}...\n")
        context += f"[{i}] {r.get('title','')}: {r.get('body','')}\n"

    print("=== AI Analysis ===\n")
    resp = client.chat.completions.create(
        model=SELECTED_MODEL,
        messages=[
            {"role": "system", "content": "Analyze search results to answer the question. Cite sources like [1], [2]. Be concise."},
            {"role": "user", "content": f"Question: {query}\n\nResults:\n{context}"}
        ],
        temperature=0.3, max_tokens=600,
    )
    print(resp.choices[0].message.content)

search_and_analyze("NVIDIA stock outlook 2026")

# Try more:
# search_and_analyze("Nifty 50 market outlook India 2026")
# search_and_analyze("IPL 2026 auction biggest buys")
# search_and_analyze("Best budget smartphones under 15000 India 2026")

Searching: "NVIDIA stock outlook 2026"

1. __symbol__ Stock Quote Price and Forecast | CNN
   2 days ago -View NVIDIA Corporation NVDA stock quote prices, financial information, real-time foreca...

2. 4 Reasons Why Nvidia Can Beat the Market Again in 2026 | The Motley Fool
   January 2, 2026 -Analysts expect revenue growth to slow to 50% in fiscal 2027, but this outlook coul...

3. NVIDIA (NVDA) Stock Forecast: Analyst Ratings, Predictions & Price Target 2026
   3 weeks ago -38 analysts have given NVIDIA (NVDA) aconsensus rating of Buywhile the NVIDIA (NVDA) pr...

4. NVIDIA (NVDA) Stock Forecast & Price Prediction 2026–2030 | CoinCodex
   3 weeks ago -In 2026, NVIDIA (NVDA) is anticipated to change hands in a trading channel between · $ ...

5. NVIDIA STOCK FORECAST 2026, 2027-2037
   3 days ago -The forecasted Nvidia price at the end of 2026 is $187- and the year to year change 0%. ...

=== AI Analysis ===

Based on the search results, the NVIDIA stock outlook for 2026 is generally 

---

## 11. Multilingual — Hindi

Open-source models like `qwen/qwen3-32b` handle Hindi and other Indian languages well.

In [119]:
# 11a. Hindi — Mutual Funds explained in Hinglish
r = client.chat.completions.create(
    model="qwen/qwen3-32b",
    messages=[
        {"role": "system", "content": "You are a friendly college senior who explains finance in Hinglish (Hindi + English mix). Keep it casual and simple."},
        {"role": "user", "content": "Mujhe samjhao ki mutual funds kya hote hain aur SIP kaise start kare? Mere paas ₹500/month spare hai — kya ye enough hai?"}
    ],
    temperature=0.6, max_tokens=500,
)
print("🇮🇳 HINDI — Mutual Funds & SIP\n")
print(r.choices[0].message.content)

🇮🇳 HINDI — Mutual Funds & SIP

<think>
Okay, the user wants to know about mutual funds and starting an SIP with ₹500/month. Let me break this down.

First, mutual funds: I need to explain them in simple terms. Maybe compare them to a group of friends investing together. They pool money and invest in stocks, bonds, etc. A professional manages it. Benefits like diversification and professional management.

Then SIP. SIP is like a recurring deposit but for mutual funds. They invest a fixed amount every month. It's good for disciplined investing and rupee cost averaging. Need to mention how to start an SIP—online platforms, choosing a fund, setting up auto-debit.

The user has ₹500/month. I should say yes, it's enough. Even small amounts can grow over time. Emphasize long-term growth and consistency. Maybe give an example: investing ₹500 monthly for 10 years at 12% returns could be around ₹1.5 lakh. But stress that returns depend on the fund's performance.

Also, mention types of mutual fu

In [120]:
# 11b. Translation — English → Hindi (Devanagari)
text = "Artificial Intelligence is transforming how we work, learn, and communicate. India is emerging as a global AI hub."

r = client.chat.completions.create(
    model="qwen/qwen3-32b",
    messages=[
        {"role": "system", "content": "You are a professional translator. Translate accurately into Hindi using Devanagari script."},
        {"role": "user", "content": f"Translate this text into Hindi (Devanagari script).\n\nText: \"{text}\""}
    ],
    temperature=0.3, max_tokens=300,
)
print("🇮🇳 HINDI TRANSLATION\n")
print(f"English: {text}\n")
print(f"Hindi:   {r.choices[0].message.content}")

🇮🇳 HINDI TRANSLATION

English: Artificial Intelligence is transforming how we work, learn, and communicate. India is emerging as a global AI hub.

Hindi:   <think>
Okay, let's tackle this translation. The user wants the given English text translated into Hindi using Devanagari script. First, I need to make sure I understand the original text correctly.

The first sentence is "Artificial Intelligence is transforming how we work, learn, and communicate." The key terms here are "Artificial Intelligence" which is commonly translated as "कृत्रिम बुद्धिमत्ता" in Hindi. Then "transforming" would be "परिवर्तित कर रही है" or "बदल रही है". The structure here is important. The sentence structure in Hindi usually follows Subject-Object-Verb, so I need to adjust accordingly.

Next part: "how we work, learn, and communicate." The verb "work" here is "काम करते हैं", "learn" is "सीखते हैं", and "communicate" is "संचार करते हैं". The phrase "how we" would be "हम कैसे" but since it's part of the sentenc

---
## 12. Multimodal — Image Understanding

`meta-llama/llama-4-scout-17b-16e-instruct` can analyze images. Pass any local image (base64 encoded).

In [121]:
# Analyze a local image (base64 encoded)
import base64

VISION_MODEL = "meta-llama/llama-4-scout-17b-16e-instruct"

def analyze_local_image(image_path, question="What's in this image? Describe it."):
    """Send a local image to Llama 4 Scout for analysis."""
    with open(image_path, "rb") as f:
        b64 = base64.b64encode(f.read()).decode("utf-8")
    ext = image_path.rsplit(".", 1)[-1].lower()
    mime = {"jpg": "image/jpeg", "jpeg": "image/jpeg", "png": "image/png", "gif": "image/gif", "webp": "image/webp"}.get(ext, "image/jpeg")
    r = client.chat.completions.create(
        model=VISION_MODEL,
        messages=[{"role": "user", "content": [
            {"type": "text", "text": question},
            {"type": "image_url", "image_url": {"url": f"data:{mime};base64,{b64}"}}
        ]}],
        temperature=0.5, max_tokens=500,
    )
    return r.choices[0].message.content

# Analyze the sample image included in the repo
result = analyze_local_image("sample_image.jpg", "Describe this image in detail. Tell us nutrition content of the food? is it healthy?")
print("🖼️ LOCAL IMAGE ANALYSIS\n")
print(result)

# Try with your own images:
# print(analyze_local_image("my_screenshot.png", "What does this screenshot show?"))
# print(analyze_local_image("my_chart.png", "Analyze this chart and identify the trend."))

🖼️ LOCAL IMAGE ANALYSIS

The image depicts a vibrant arrangement of fresh vegetables and a bowl of salad on a dark gray surface. The composition includes:

* A bowl of salad with mixed ingredients, including peas, potatoes, tomatoes, carrots, and herbs
* A bunch of parsley tied with twine
* Several cherry tomatoes
* Two green bell peppers
* Two red bell peppers
* Two carrots

**Nutrition Content:**

The nutrition content of the food in the image is not explicitly provided, but based on the visible ingredients, we can make some general observations:

* The salad appears to contain a variety of vegetables, including:
	+ Peas (rich in protein, fiber, and vitamins)
	+ Potatoes (good source of complex carbohydrates, fiber, and potassium)
	+ Tomatoes (high in vitamin C and lycopene)
	+ Carrots (rich in vitamin A and fiber)
	+ Bell peppers (high in vitamin C and antioxidants)
* The presence of parsley suggests that the salad may also contain a boost of vitamin K and antioxidants.

**Healthine

---
## Summary

| # | Section | Model Used | Highlights |
|---|---------|------------|------------|
| 1-2 | Setup & Models | — | 12+ free models from Meta, OpenAI, Alibaba, Moonshot |
| 3 | Basic Chat | `llama-3.3-70b-versatile` | OpenAI-compatible API basics |
| 4 | Sentiment | `llama-3.3-70b-versatile` | NVIDIA headline scoring |
| 5 | Investment Thesis | `llama-3.3-70b-versatile` | Reliance Industries analysis |
| 6 | Model Comparison | 6 models | Same prompt — compare speed & quality |
| 7 | Streaming + JSON | `llama-3.3-70b-versatile` | Real-time output & structured extraction |
| 8 | Batch Ratings | `llama-3.1-8b-instant` | 6 stocks rated in seconds |
| 9 | Fun Use Cases | `llama-3.3-70b-versatile` | Cricket, startup pitch, exam prep |
| 10 | Web Search + LLM | `llama-3.3-70b-versatile` | DuckDuckGo grounded analysis |
| 11 | Multilingual | `qwen/qwen3-32b` | Hindi (Hinglish) + Hindi translation |
| 12 | Multimodal | `llama-4-scout-17b` | Local image analysis |

### Try Experimenting!
1. Change `SELECTED_MODEL` and re-run — compare quality across models
2. Adjust `temperature` (0.0 = factual, 1.0 = creative)
3. Replace stock data with your favorite companies
4. Try `qwen/qwen3-32b` for Hindi and other languages
5. Try `meta-llama/llama-4-scout-17b-16e-instruct` with your own images
6. Modify the DuckDuckGo queries to search anything in real-time